例えば、大学前のアパートを入力とし、そこから20分で移動できる場所を取得したい。以下の合計を求めることになる
・入力から徒歩20分のエリア
・入力から20分以内に到達可能な交通機関から(20分-駅までの時間)分で到着できる別の駅とそこから(20分-(駅までの時間+別の駅までの乗車時間))分以内で到達できるエリア、再帰の可能性あり  
手順
1. 20分以内の徒歩エリアを求める
2. 20分以内で到達できる徒歩での乗り換えが必要ない交通機関を求める(以下ダイレクトと呼称)
3. (ダイレクトまでの徒歩時間と、ダイレクトまでの乗車時間を引いた)時間以内の、ダイレクトからの徒歩エリアを取得
4. 2, 3を繰り返す.
5. すべてを結合

3,4を作成

In [7]:
from dotenv import load_dotenv

from engine import search_nearby_walking_distance_stations, get_stations_contain_area, get_same_line_or_route_stations, \
    get_route_yahoo_transit, get_same_line_or_route_stations_with_time
from engine.mapbox import MapBoxApi, IsochroneProfile, concat_isochrones, isochrone_to_str
import os
from engine.bus import *
from engine.train import *
from collections import deque

# データの読み込み
dataset: dict[TransitType, list[Station]] = {
    TransitType.BUS: load_stop_data("../dataset/busstops/kanagawa/P11-22_14.geojson"),
    TransitType.TRAIN: load_station_data("../dataset/stations/N02-20_Station.geojson"),
}
load_dotenv()
mapbox_api = MapBoxApi(os.getenv("MAPBOX_API_TOKEN"))

In [8]:
from datetime import datetime


def _search_reachable_area(
        api: MapBoxApi, dataset: dict[TransitType, list[Station]],
        start_point: Coordinate, time_limit: int, transit_types: list[TransitType]) -> tuple[list[dict], list[tuple[int, Station]]]:
    # reachable_areas = {tl: [] for tl in time_limits} # { int(tl1): [] }で初期化
    reachable_area:list[dict] = []
    reachable_stations: list[tuple[int, Station]] = []

    # (1) 入力位置の周辺
    start_time = datetime.now()
    walking_distance_area = api.get_isochrone(
        prof=IsochroneProfile.Walking,
        coordinate=start_point,
        contours_minutes=[time_limit]
    )
    end_time = datetime.now()
    print(f"[1] Get Walking Distance Isochrone: Done: {end_time-start_time}")
    
    # 徒歩圏内に含まれる交通機関一覧
    start_time = datetime.now()
    _stations = get_stations_contain_area(
        dataset=dataset,
        isochrone=walking_distance_area,
        transit_types=transit_types,
    )
    end_time = datetime.now()
    print(f"[1] Get Stations within Isochrone: Done: {end_time-start_time}")
    
    # 各交通機関まで徒歩でかかる時間を計算
    print("[1] Get Station Travel time: Running...")
    start_time=datetime.now()
    stations_with_travel_time: list[tuple[int, Station]] = []
    for station in _stations:
        travel_time = api.get_walking_travel_time(start_point, station.geometry.calc_mean())
        stations_with_travel_time.append(
            (travel_time, station)
        )
        # print(f"  - {station.name}: {travel_time}mins")
    end_time=datetime.now()
    print(f"[1] Get Station Travel time: Done: {end_time-start_time}")
    
    # (2) 徒歩圏内の交通機関を使って時間以内に行ける別の駅を表示する
    print("[2] Get Stations from Result of [1]: Running...")
    start_time = datetime.now()
    for walking_travel_time, station in stations_with_travel_time:
        # (最大移動時間time_limit-各交通機関への移動時間)分以内に辿り着ける同じ路線の駅を取得
        same_line_stations_with_travel_time = get_same_line_or_route_stations_with_time(station, dataset, time_limit-walking_travel_time)
        for transit_travel_time, same_line_station in same_line_stations_with_travel_time:
            reachable_stations.append(
                # 交通機関までの移動時間とそこから交通機関をつかった時間
                (walking_travel_time+transit_travel_time, same_line_station)
            )
    end_time = datetime.now()
    print(f"[2] Get Stations from Result of [1]: Done: {end_time-start_time}")

    # # 移動先の交通機関からtime_limit以内に行ける範囲
    # print("[2] Isochrone from Result of [2]: Running...")
    # for travel_time, reachable_station in reachable_stations:
    #     remaining_time = time_limit-travel_time
    #     if 60 >= remaining_time >= 1:
    #         isochrone = api.get_isochrone(
    #             prof=IsochroneProfile.Walking,
    #             coordinate=reachable_station.geometry.calc_mean(),
    #             contours_minutes=[remaining_time]
    #         )
    #         reachable_area.append(
    #             isochrone
    #         )
    # print("[2] Isochrone from Result of [2]: Done")
    
    return reachable_area, reachable_stations
def  search_reachable_area(
        api: MapBoxApi, dataset: dict[TransitType, list[Station]],
        start_point: Coordinate, time_limit: int, transit_types: list[TransitType]) -> tuple[list[dict], list[tuple[int, Station]]]:
    reachable_areas:list[dict] = []
    reachable_stations: list[tuple[int, Station]] = []
    
    already_checked: list[Coordinate] = []
    
    queue = deque([(0, start_point)])
    while len(queue) != 0:
        print(f"[SR] Total waiting: {len(queue)}")
        val = queue.popleft()
        reachable_travel_time, coordinate = val
        if coordinate in already_checked:
            continue
        already_checked.append(coordinate)
        print(f"  - [SR] {reachable_travel_time}, {coordinate.to_folium()}")
        
        can_use_travel_time = time_limit-reachable_travel_time
        got_areas, got_stations_with_time = _search_reachable_area(api, dataset, coordinate, can_use_travel_time, transit_types)
        # かぶってなければ格納
        for area in got_areas:
            if area not in reachable_areas:
                reachable_areas.append(area)
        for travel_time_station_pair  in got_stations_with_time:
            if travel_time_station_pair not in reachable_stations:
                reachable_stations.append(travel_time_station_pair)
                tt, stat = travel_time_station_pair
                if tt < can_use_travel_time:
                    queue.append((tt, stat.geometry.calc_mean()))        
        
    return reachable_areas, reachable_stations


In [9]:
# import folium
# from backend.engine.mapbox import concat_isochrones
# m = folium.Map(
#     location=[35.4861002, 139.3399782],
#     zoom_start=14,
#     tiles="https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png",
#     attr=f"出典: 国土地理院ウェブサイト・地理院タイル・標準地図 {'(C) MAPBOX'}",
# )
# 
# # print(reachable_area)
# 
# # print(reachable_area[60])
# folium.GeoJson(reachable_area).add_to(m)
# # print(reachable_area)
# # for isochrone in reachable_area:
# #     print(isochrone)
# #     folium.GeoJson(isochrone).add_to(m)

# folium.GeoJson(concat_isochrones(reachable_area).to_json()).add_to(m)   

# m

In [10]:
def search_nearby_stations(api: MapBoxApi, start_point: Coordinate, time_limit: int, station_transit_types: list[TransitType]) ->list[tuple[int, Station]]:
    walking_distance_area = api.get_isochrone(
        prof=IsochroneProfile.Walking,
        coordinate=start_point,
        contours_minutes=[time_limit]
    )
    nearby_stations = get_stations_contain_area(
        dataset=dataset,
        isochrone=walking_distance_area,
        transit_types=station_transit_types,
    )
    
    nearby_stations_with_travel_time: list[tuple[int, Station]] = []
    for nearby_station in nearby_stations:
        travel_time_from_start_point = api.get_walking_travel_time(
            start_point,
            nearby_station.geometry.calc_mean()
        )
        nearby_stations_with_travel_time.append(
            (travel_time_from_start_point, nearby_station)
        )
    
    return nearby_stations_with_travel_time

In [11]:
from engine import get_reachable_stations

# 厚木市役所を入力とする.
input_coordinate = Coordinate(Lat=35.4429973, Lng=139.3611488)
# 上記まで30分で行ける範囲を検索する
input_time_limit = 30

# nearby_stations_from_input = search_nearby_stations(
#     api=mapbox_api,
#     start_point=input_coordinate,
#     time_limit=input_time_limit,
#     station_transit_types=[TransitType.BUS, TransitType.TRAIN]
# )



print("厚木市役所から")
nearby_stations = get_reachable_stations(
    api=mapbox_api,
    dataset=dataset,
    start_point=input_coordinate,
    time_limit=input_time_limit,
    transit_types=[TransitType.BUS, TransitType.TRAIN]
)

for travel_time, nearby_station in nearby_stations:
    print(f"{travel_time}分: {''.join(nearby_station.management_groups)}-{nearby_station.name}")
    
    
    

厚木市役所から
queue: 1
queue: 252
queue: 251
queue: 250
queue: 249
queue: 248
queue: 247
queue: 246
queue: 245
queue: 244
queue: 243
queue: 242
queue: 241
queue: 240
queue: 239
queue: 238
queue: 237
queue: 236
queue: 235
queue: 234
queue: 233
queue: 232
queue: 231
queue: 230
queue: 229
queue: 228
queue: 227
queue: 226
queue: 225
queue: 226
queue: 225
queue: 224
queue: 223
queue: 222
queue: 221
queue: 220
queue: 219
queue: 218
queue: 217
queue: 216
queue: 215
queue: 214
queue: 227
queue: 226
queue: 225
queue: 224
queue: 223
queue: 222
queue: 221
queue: 220
queue: 219
queue: 218
queue: 217
queue: 216
queue: 215
queue: 214
queue: 213
queue: 212
queue: 211
queue: 210
queue: 209
queue: 208
queue: 207
queue: 206
queue: 205
queue: 204
queue: 203
queue: 202
queue: 201
queue: 200
queue: 199
queue: 198
queue: 197
queue: 196
queue: 204
queue: 203
queue: 202
queue: 201
queue: 200
queue: 199
queue: 198
queue: 197
queue: 196
queue: 195
queue: 194
queue: 193
queue: 192
queue: 191
queue: 190
queue: 189
queu